# 05 – Integration and Demo
**Movie Recommendation System | Phase 5**

This notebook joins the outputs of the four earlier phases into one working pipeline:

| Phase | Notebook | What it hands over | Used here |
|---|---|---|---|
| 1 EDA | `01_data_exploration` | Understanding of the data (sparsity, long tail, cold start) | Context for the design choices |
| 2 Cleaning | `02_data_cleaning` | `data/processed/*.csv` | Loaded below |
| 3 Modelling | `03_model_development` | Tuned SVD settings, top-5 rules (min. 30 ratings, skip rated movies) | Re-used in `src/recommender.py` |
| 4 Evaluation | `04_model_evaluation` | Accuracy, ranking and variety results, known limits | Summarised in section 3 |

Run from the `notebooks/` folder or the repo root; the path is resolved below.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 60)
from src.recommender import MovieRecommender, MIN_MOVIE_RATINGS, MIN_USER_RATINGS, SVD_PARAMS
print('Repo root:', ROOT)

Repo root: /home/claude/movie-recommender


## 1. Check that the stage outputs fit together

In [2]:
P = ROOT / 'data' / 'processed'
train, test, full = (pd.read_csv(P / f) for f in ('train.csv', 'test.csv', 'ratings_clean.csv'))
movies = pd.read_csv(P / 'movies_clean.csv')

checks = {
    'train + test = cleaned ratings': len(train) + len(test) == len(full),
    'no (user, movie) pair in both train and test': not (set(zip(train.userId, train.movieId)) & set(zip(test.userId, test.movieId))),
    'all rated movies exist in the catalogue': set(full.movieId) <= set(movies.movieId),
    'all 610 users appear in train and test': train.userId.nunique() == test.userId.nunique() == 610,
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
print(f'\ntrain {len(train):,} | test {len(test):,} | all ratings {len(full):,} | movies {len(movies):,}')

PASS - train + test = cleaned ratings
PASS - no (user, movie) pair in both train and test
PASS - all rated movies exist in the catalogue
PASS - all 610 users appear in train and test

train 80,672 | test 20,164 | all ratings 100,836 | movies 9,742


## 2. The model that is deployed
Phase 3 chose a tuned **SVD** (collaborative filtering on ratings only). The settings are copied unchanged:

In [3]:
SVD_PARAMS, MIN_MOVIE_RATINGS

({'n_factors': 150,
  'n_epochs': 30,
  'lr_all': 0.01,
  'reg_all': 0.1,
  'random_state': 42},
 30)

For the live system the model is fitted on **all** cleaned ratings (`ratings_clean.csv`) plus the new user's ratings. 
`train.csv` / `test.csv` were only needed for honest evaluation in Phase 4 and are not used by the interface.

## 3. What Phase 4 found (read this before showing the system to anyone)
Figures below are quoted from `04_model_evaluation.ipynb`.

| Question | Result |
|---|---|
| Rating accuracy on held-out test set | RMSE **0.8745**, MAE 0.667 (global mean 1.050; BaselineOnly 0.884) |
| Is the top-5 list good? | About 6x better than random (NDCG@5 0.100 vs 0.017) but **below a plain "most popular" list** (0.160) |
| Variety | Only **134 distinct movies** across 610 users; ~65 % of users get the same #1 film |
| Cold start | RMSE about 1.0 for users with 25 or fewer ratings and for unseen movies |

So the interface asks for a minimum of 5 ratings and recommends 15 or more, and the documentation states these limits openly.

## 4. Receiving user ratings
A user's input is a dictionary `{movieId: rating}`. `rec.search_movies()` and `rec.popular_movies()` help find IDs; `validate_ratings()` rejects bad input.

In [4]:
rec = MovieRecommender()
rec.search_movies('toy story', limit=3)

,movieId,title,genres,n_ratings
0,1,Toy Story (1995),"Adventure, Animation, Children, Comedy, Fantasy",215
1,3114,Toy Story 2 (1999),"Adventure, Animation, Children, Comedy, Fantasy",97
2,78499,Toy Story 3 (2010),"Adventure, Animation, Children, Comedy, Fantasy, IMAX",55


In [5]:
from src.recommender import InvalidRatingError
for bad in ({1: 5.0}, {1: 5, 2: 5, 3: 5, 4: 5, 5: 3.3}):
    try:
        rec.recommend(bad)
    except InvalidRatingError as e:
        print('Rejected:', e)

Rejected: Please rate at least 5 movies (got 1); with fewer the model cannot learn your taste.
Rejected: Rating 3.3 for movieId 5 must be 0.5 to 5.0 in half-star steps


## 5. Connect the input to the model and show the top five

In [6]:
my_ratings = {
    2571: 5.0,   # The Matrix
    260:  4.5,   # Star Wars: A New Hope
    318:  5.0,   # The Shawshank Redemption
    296:  4.0,   # Pulp Fiction
    1:    4.5,   # Toy Story
    111:  3.5,   # Taxi Driver
}
recs = rec.recommend(my_ratings, n=5)
recs[['title', 'genres', 'predicted_rating']]

,title,genres,predicted_rating
rank,,,
1,"Boondock Saints, The (2000)","Action, Crime, Drama, Thriller",4.44
2,Rear Window (1954),"Mystery, Thriller",4.43
3,In Bruges (2008),"Comedy, Crime, Drama, Thriller",4.42
4,"Usual Suspects, The (1995)","Crime, Mystery, Thriller",4.42
5,Life Is Beautiful (La Vita è bella) (1997),"Comedy, Drama, Romance, War",4.41


## 6. Run it yourself
* Command line: `python cli.py` (or `python cli.py --demo`)
* Web app: `streamlit run app.py`